In [ ]:
import os

# Uninstall any existing pyspark to prevent version conflicts
!pip uninstall pyspark -y -q

# Install the desired pyspark version
!pip install pyspark==3.5.1 -q

# Install graphframes and findspark
!pip install graphframes -q
!pip install findspark -q

import findspark
# Initialize findspark to set SPARK_HOME and ensure the correct Spark binaries are used
findspark.init()

from google.colab import drive
drive.mount('/content/drive')
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc, udf, unix_timestamp, to_timestamp

# Explicitly configure the SparkSession to use the graphframes package compatible with Spark 3.5
spark = (
    SparkSession.builder
    .appName("Lab6_GraphFrames")
    .config("spark.jars.packages",
            "graphframes:graphframes:0.8.3-spark3.5-s_2.12")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setCheckpointDir("/tmp/spark-checkpoints")

from graphframes import GraphFrame

print("Spark version :", spark.version)
print("Setup complete!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark version : 3.5.1
Setup complete!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# the station df as vertex table
station_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/content/drive/MyDrive/ESI TPs/BigData/Data/TP6/station_data.csv")
)

# trip df as edge table
trip_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/content/drive/MyDrive/ESI TPs/BigData/Data/TP6/trip_data.csv")
)

print("StationDf")
station_df.printSchema()
station_df.show(3)

print("TripDf")
trip_df.printSchema()
trip_df.show(3)

StationDf
root
 |-- station_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dockcount: integer (nullable = true)
 |-- landmark: string (nullable = true)
 |-- installation: string (nullable = true)

+----------+--------------------+---------+-----------+---------+--------+------------+
|station_id|                name|      lat|       long|dockcount|landmark|installation|
+----------+--------------------+---------+-----------+---------+--------+------------+
|         2|San Jose Diridon ...|37.329732|-121.901782|       27|San Jose|    8/6/2013|
|         3|San Jose Civic Ce...|37.330698|-121.888979|       15|San Jose|    8/5/2013|
|         4|Santa Clara at Al...|37.333988|-121.894902|       11|San Jose|    8/6/2013|
+----------+--------------------+---------+-----------+---------+--------+------------+
only showing top 3 rows

TripDf
root
 |-- Trip ID: integer (nullable = true)
 |-- Duration: 

In [ ]:

vertices = station_df.withColumnRenamed("name", "id")


vertices.show(5)
print("Total vertices:", vertices.count())

+----------+--------------------+---------+-----------+---------+--------+------------+
|station_id|                  id|      lat|       long|dockcount|landmark|installation|
+----------+--------------------+---------+-----------+---------+--------+------------+
|         2|San Jose Diridon ...|37.329732|-121.901782|       27|San Jose|    8/6/2013|
|         3|San Jose Civic Ce...|37.330698|-121.888979|       15|San Jose|    8/5/2013|
|         4|Santa Clara at Al...|37.333988|-121.894902|       11|San Jose|    8/6/2013|
|         5|    Adobe on Almaden|37.331415|  -121.8932|       19|San Jose|    8/5/2013|
|         6|    San Pedro Square|37.336721|-121.894074|       15|San Jose|    8/7/2013|
+----------+--------------------+---------+-----------+---------+--------+------------+
only showing top 5 rows

Total vertices: 72


In [ ]:
edges = (
    trip_df
    .withColumnRenamed("Start Station", "src")
    .withColumnRenamed("End Station", "dst")
)


edges.select("src", "dst", "Duration", "Bike #", "Start Date", "End Date").show(5)
print(edges.count())

+--------------------+--------------------+--------+------+---------------+---------------+
|                 src|                 dst|Duration|Bike #|     Start Date|       End Date|
+--------------------+--------------------+--------+------+---------------+---------------+
|Harry Bridges Pla...|San Francisco Cal...|     765|   288|8/31/2015 23:26|8/31/2015 23:39|
|San Antonio Shopp...|Mountain View Cit...|    1036|    35|8/31/2015 23:11|8/31/2015 23:28|
|      Post at Kearny|   2nd at South Park|     307|   468|8/31/2015 23:13|8/31/2015 23:18|
|  San Jose City Hall| San Salvador at 1st|     409|    68|8/31/2015 23:10|8/31/2015 23:17|
|Embarcadero at Fo...|Embarcadero at Sa...|     789|   487|8/31/2015 23:09|8/31/2015 23:22|
+--------------------+--------------------+--------+------+---------------+---------------+
only showing top 5 rows

354154


In [ ]:
g = GraphFrame(vertices, edges)

print("Number of vertices:", g.vertices.count())
print("Number of edges:",    g.edges.count())


g.vertices.show(3)
print("Sample edges:")
g.edges.select("src", "dst", "Duration").show(3)

Number of vertices: 72
Number of edges: 354154
+----------+--------------------+---------+-----------+---------+--------+------------+
|station_id|                  id|      lat|       long|dockcount|landmark|installation|
+----------+--------------------+---------+-----------+---------+--------+------------+
|         2|San Jose Diridon ...|37.329732|-121.901782|       27|San Jose|    8/6/2013|
|         3|San Jose Civic Ce...|37.330698|-121.888979|       15|San Jose|    8/5/2013|
|         4|Santa Clara at Al...|37.333988|-121.894902|       11|San Jose|    8/6/2013|
+----------+--------------------+---------+-----------+---------+--------+------------+
only showing top 3 rows

Sample edges:
+--------------------+--------------------+--------+
|                 src|                 dst|Duration|
+--------------------+--------------------+--------+
|Harry Bridges Pla...|San Francisco Cal...|     765|
|San Antonio Shopp...|Mountain View Cit...|    1036|
|      Post at Kearny|   2nd at S

In [ ]:

#5) Return the number of trips made between each source and destination, sorted in
#descending order (based on the number of trips).
trips_per_route = (
  g.edges
  .groupBy("src", "dst").count()
  .orderBy("count", ascending=False)

)

trips_per_route.show(10)

+--------------------+--------------------+-----+
|                 src|                 dst|count|
+--------------------+--------------------+-----+
|San Francisco Cal...|     Townsend at 7th| 3748|
|Harry Bridges Pla...|Embarcadero at Sa...| 3145|
|     2nd at Townsend|Harry Bridges Pla...| 2973|
|     Townsend at 7th|San Francisco Cal...| 2734|
|Harry Bridges Pla...|     2nd at Townsend| 2640|
|Embarcadero at Fo...|San Francisco Cal...| 2439|
|   Steuart at Market|     2nd at Townsend| 2356|
|Embarcadero at Sa...|   Steuart at Market| 2330|
|     Townsend at 7th|San Francisco Cal...| 2192|
|Temporary Transba...|San Francisco Cal...| 2184|
+--------------------+--------------------+-----+
only showing top 10 rows



In [ ]:
# 6) Return the number of trips that start or end at the station "Townsend at 7th",
# sorted in descending order(based on the number of trips).
townsend_trips = (
g.edges
  .where("src = 'Townsend at 7th' OR dst = 'Townsend at 7th'")
  .groupBy("src", "dst").count()
  .orderBy("count", ascending=False)
)

townsend_trips.show(10)

+--------------------+--------------------+-----+
|                 src|                 dst|count|
+--------------------+--------------------+-----+
|San Francisco Cal...|     Townsend at 7th| 3748|
|     Townsend at 7th|San Francisco Cal...| 2734|
|     Townsend at 7th|San Francisco Cal...| 2192|
|     Townsend at 7th|Civic Center BART...| 1844|
|Civic Center BART...|     Townsend at 7th| 1765|
|San Francisco Cal...|     Townsend at 7th| 1198|
|Temporary Transba...|     Townsend at 7th|  834|
|     Townsend at 7th|Harry Bridges Pla...|  827|
|   Steuart at Market|     Townsend at 7th|  746|
|     Townsend at 7th|Temporary Transba...|  740|
+--------------------+--------------------+-----+
only showing top 10 rows



In [ ]:
# 7)Return the vertices that have never been a destination for a trip starting from "Spear at Folsom"
reached = (
    g.edges
    .where("src = 'Spear at Folsom'")
    .selectExpr("dst as id")
    .distinct()
)

# All station ids
all_stations = g.vertices.select("id")

# Stations never reached = all stations minus reached stations
never_reached = all_stations.subtract(reached)
never_reached.show(10)

+--------------------+
|                  id|
+--------------------+
|           Japantown|
|Arena Green / SAP...|
|San Antonio Shopp...|
|Cowper at University|
|         FarStationA|
|Paseo de San Antonio|
|  San Jose City Hall|
|       St James Park|
|Redwood City Calt...|
|San Antonio Caltr...|
+--------------------+
only showing top 10 rows



In [ ]:
#8) Return the station with the maximum number of incoming trips.
# inDegrees returns a DataFrame with columns: id, inDegree


max_incoming = g.inDegrees.orderBy("inDegree", ascending=False).limit(1)
max_incoming.show()

+--------------------+--------+
|                  id|inDegree|
+--------------------+--------+
|San Francisco Cal...|   34810|
+--------------------+--------+



In [ ]:
#9) Return the trip with the longest duration.
longest_trip = (
    g.edges
    .select("Trip ID", "src", "dst", "Duration", "Bike #", "Start Date", "End Date")
    .orderBy(desc("Duration"))
    .limit(1)
)

longest_trip.show(truncate=False)

+-------+------------------------+-------------+--------+------+---------------+---------------+
|Trip ID|src                     |dst          |Duration|Bike #|Start Date     |End Date       |
+-------+------------------------+-------------+--------+------+---------------+---------------+
|568474 |South Van Ness at Market|2nd at Folsom|17270400|535   |12/6/2014 21:59|6/24/2015 20:18|
+-------+------------------------+-------------+--------+------+---------------+---------------+



In [ ]:
#10) Write a query that finds the cycle of three trips made by the same bike with the shortest time duration.



triangles = g.find("(a)-[e1]->(b); (b)-[e2]->(c); (c)-[e3]->(a)")# here we get all cycle of three trip


shortest_cycle = triangles \
    .filter("e1.`Bike #` = e2.`Bike #` AND e2.`Bike #` = e3.`Bike #`").selectExpr(
        "a.id as station_A",
        "b.id as station_B",
        "c.id as station_C",
        "e1.`Bike #` as bike_id",
        "e1.Duration + e2.Duration + e3.Duration as total_duration"
    ) \
    .orderBy("total_duration", ascending=True) \
    .limit(1)

shortest_cycle.show(truncate=False)

+-----------------+-----------------+-----------------+-------+--------------+
|station_A        |station_B        |station_C        |bike_id|total_duration|
+-----------------+-----------------+-----------------+-------+--------------+
|Market at Sansome|Market at Sansome|Market at Sansome|109    |180           |
+-----------------+-----------------+-----------------+-------+--------------+



### Detect abnormal bike travel speed between stations: Write a GraphFrames query
to detect suspiciously fast bike movements. Find sequences of two consecutive trips
made by the same bike where:
-  the second trip starts within 2 minutes after the first trip ends
- the two stations belong to different cities (landmark)
- the distance between stations is greater than 10 km
- > Return the bike number and the stations involved.
- > This detects physically impossible bike relocations

In [ ]:
print("=== Q11: Suspiciously fast bike movements (Professor Style) ===")

#  Two consecutive trips
paths = g.find("(a)-[e1]->(b); (b)-[e2]->(c)")

suspicious_trips = paths \
    .filter("e1.`Bike #` = e2.`Bike #`") \
    .filter("b.landmark != c.landmark") \
    .filter("unix_timestamp(e2.`Start Date`, 'M/d/yyyy H:mm') - unix_timestamp(e1.`End Date`, 'M/d/yyyy H:mm') <= 120") \
    .filter("sqrt(pow(b.lat - c.lat, 2) + pow(b.long - c.long, 2)) * 111 > 10") \
    .selectExpr(
        "e1.`Bike #` as bike_id",
        "b.id as start_station",
        "c.id as end_station",
        "b.landmark as city_1",
        "c.landmark as city_2"
    )

suspicious_trips.show(truncate=False)

=== Q11: Suspiciously fast bike movements (Professor Style) ===
+-------+------------------------------+------------------------------+-------------+-------------+
|bike_id|start_station                 |end_station                   |city_1       |city_2       |
+-------+------------------------------+------------------------------+-------------+-------------+
|249    |MLK Library                   |Mountain View Caltrain Station|San Jose     |Mountain View|
|146    |Mountain View Caltrain Station|Palo Alto Caltrain Station    |Mountain View|Palo Alto    |
|149    |Mountain View Caltrain Station|University and Emerson        |Mountain View|Palo Alto    |
|149    |Mountain View Caltrain Station|University and Emerson        |Mountain View|Palo Alto    |
|149    |Mountain View Caltrain Station|University and Emerson        |Mountain View|Palo Alto    |
|149    |Mountain View Caltrain Station|University and Emerson        |Mountain View|Palo Alto    |
|149    |Mountain View Caltrain Stat

In [ ]:
##12)Create a subgraph that
#contains only the trips that start or end at "Townsend at 7th".
#edges => trips
#Nodes=> station

sub_edges = g.edges.filter("src = 'Townsend at 7th' OR dst = 'Townsend at 7th'")

#  Collect all
src_ids = sub_edges.selectExpr("src as id")
dst_ids = sub_edges.selectExpr("dst as id")

# Combine them and remove duplicates
involved_ids = src_ids.union(dst_ids).distinct()

#  Filter the original vertices to only keep the involved ones
sub_vertices = g.vertices.join(involved_ids, "id", "inner")


subgraph = GraphFrame(sub_vertices, sub_edges)


subgraph.vertices.show(truncate=False)

+---------------------------------------------+----------+---------+-----------+---------+-------------+------------+
|id                                           |station_id|lat      |long       |dockcount|landmark     |installation|
+---------------------------------------------+----------+---------+-----------+---------+-------------+------------+
|5th at Howard                                |57        |37.781752|-122.405127|15       |San Francisco|8/21/2013   |
|Howard at 2nd                                |63        |37.786978|-122.398108|19       |San Francisco|8/22/2013   |
|Commercial at Montgomery                     |45        |37.794231|-122.402923|15       |San Francisco|8/19/2013   |
|Civic Center BART (7th at Market)            |72        |37.781039|-122.411748|23       |San Francisco|8/23/2013   |
|2nd at Folsom                                |62        |37.785299|-122.396236|19       |San Francisco|8/22/2013   |
|Clay at Battery                              |41       

In [ ]:
from graphframes import GraphFrame

print("=== Q13: All triangle patterns (Sampled to run fast) ===")

fast_edges = g.edges.sample(withReplacement=False, fraction=0.1, seed=42)

small_g = GraphFrame(g.vertices, fast_edges)

triangles = small_g.find("(a)-[e1]->(b); (b)-[e2]->(c); (c)-[e3]->(a)")

unique_triangles = triangles \
    .filter("a.id < b.id AND b.id < c.id") \
    .selectExpr(
        "a.id as station_A",
        "b.id as station_B",
        "c.id as station_C"
    ) \
    .distinct()

unique_triangles.show(10, truncate=False)

=== Q13: All triangle patterns (Sampled to run fast) ===
+----------------------+---------------------------------------+---------------------------------------------+
|station_A             |station_B                              |station_C                                    |
+----------------------+---------------------------------------+---------------------------------------------+
|Embarcadero at Sansome|Steuart at Market                      |Yerba Buena Center of the Arts (3rd @ Howard)|
|Embarcadero at Sansome|Steuart at Market                      |Temporary Transbay Terminal (Howard at Beale)|
|Market at 4th         |San Francisco Caltrain 2 (330 Townsend)|Temporary Transbay Terminal (Howard at Beale)|
|Market at 4th         |San Francisco Caltrain 2 (330 Townsend)|Spear at Folsom                              |
|Market at Sansome     |Powell Street BART                     |San Francisco Caltrain (Townsend at 4th)     |
|Market at Sansome     |Powell Street BART             

In [ ]:
##14) Return all paths that pass through three vertices and start from "Townsend at 7th".


three_hop_paths = g.find("(a)-[e1]->(b); (b)-[e2]->(c)")

paths_from_townsend = three_hop_paths \
    .filter(f"a.id = 'Townsend at 7th'") \
    .selectExpr(
        "a.id as start",
        "b.id as intermediate",
        "c.id as end"
    ) \
    .distinct()

paths_from_townsend.show(15, truncate=False)

+---------------+---------------------------------+----------------------------------------+
|start          |intermediate                     |end                                     |
+---------------+---------------------------------+----------------------------------------+
|Townsend at 7th|Civic Center BART (7th at Market)|Beale at Market                         |
|Townsend at 7th|Howard at 2nd                    |San Francisco Caltrain (Townsend at 4th)|
|Townsend at 7th|5th at Howard                    |San Francisco Caltrain 2 (330 Townsend) |
|Townsend at 7th|Howard at 2nd                    |2nd at South Park                       |
|Townsend at 7th|Commercial at Montgomery         |San Francisco City Hall                 |
|Townsend at 7th|Howard at 2nd                    |San Francisco Caltrain 2 (330 Townsend) |
|Townsend at 7th|Howard at 2nd                    |Market at 4th                           |
|Townsend at 7th|5th at Howard                    |Townsend at 7th    